# Amazon ML Challenge 2026 — Business Entity Resolution: full pipeline on AWS EC2

This notebook runs the **complete pipeline on one EC2 instance, in parallel with the local run**:

1. **Base training** — normalize, block 180k train S1 (test-like: 19% of S1 removed), featurize, stage-1 LightGBM, stage 2.
2. **Big-data training (v10/v11)** — adds 600k–900k extra training S1 (all positives + hard negatives, easy negatives sampled), stage 1 on everything, **3-fold out-of-fold** stage 2 (the step the 16 GB laptop could not finish).
3. **Test inference** — block all 1.73M test S1 against 10M S2/S3 records, score 78M candidate pairs, stage 2.
4. **Leaderboard decision rule** — extra matches need p ≥ 0.82, each S1's single best candidate needs p ≥ 0.55 (the rule that scored **0.980496** on the public LB), plus the official validator.

Everything is launched with `nohup` from one driver script, so **closing the browser or losing SSH does not stop it**; re-running the launch cell resumes from the last finished stage.

---
## Instance recommendation

| Instance | vCPU / RAM | ≈ $/h (us-east-1 on-demand) | Verdict |
|---|---|---|---|
| c7i.4xlarge | 16 / 32 GB | ~0.71 | works with `N_SHARDS=2`, but RAM is tight and it is ~2× slower |
| **m7i.8xlarge** | **32 / 128 GB** | **~1.61** | **recommended** — no swapping, `N_SHARDS=3` (900k extra S1); full run ≈ 3.5–4.5 h ≈ $6–7 |
| r7i.4xlarge | 16 / 128 GB | ~1.06 | budget: plenty of RAM, fewer cores (≈ 5–6 h) |

* **AMI:** Ubuntu Server 24.04 LTS (x86_64). **Storage:** 100 GB gp3 EBS.
* **Security group:** SSH (22) from your IP only. Reach Jupyter through an SSH tunnel (below) — do not open port 8888.
* If launching a 32-vCPU instance fails, your On-Demand vCPU quota is < 32 → use r7i.4xlarge or request an increase
  (*Service Quotas → Amazon EC2 → Running On-Demand Standard (A, C, D, H, I, M, R, T, Z) instances*).
* **Stop the instance when the run finishes** (last cell) — billing continues while it runs.

---
## One-time setup (from your Windows PowerShell)

```powershell
# 1. copy the code bundle + dataset to the instance (replace KEY and IP)
cd "C:\Users\surya\Downloads\amazon ml"
Compress-Archive -Path dataset -DestinationPath dataset.zip -Force        # ~1 GB zip, once
scp -i KEY.pem aws_code_bundle.zip dataset.zip AWS_EC2_run_pipeline.ipynb ubuntu@IP:~/

# 2. log in with a tunnel for Jupyter
ssh -i KEY.pem -L 8888:localhost:8888 ubuntu@IP
```
On the instance (first login only):
```bash
sudo apt-get update && sudo apt-get install -y python3-venv python3-pip unzip
python3 -m venv ~/venv && source ~/venv/bin/activate
pip install jupyterlab ipykernel && python -m ipykernel install --user --name er --display-name "ER (venv)"
jupyter lab --no-browser --port 8888      # open the printed http://localhost:8888/... link on your PC
```
Open this notebook in Jupyter, select the **ER (venv)** kernel, and run the cells top to bottom.

## 1. Hardware check

In [ ]:
!nproc; free -g; df -h ~ | tail -1; python3 --version

## 2. Python environment (pinned versions from `requirements.txt`)

In [ ]:
%%bash
source ~/venv/bin/activate
cd ~
mkdir -p amazon_ml && cd amazon_ml
[ -d code ] || unzip -q -o ~/aws_code_bundle.zip -d .
pip install -q -r code/business_entity_resolution/requirements.txt
python -c "import polars, numba, lightgbm, rapidfuzz, pyarrow, numpy; print('polars', polars.__version__, '| numba', numba.__version__, '| lightgbm', lightgbm.__version__, '| rapidfuzz', rapidfuzz.__version__)"

## 3. Data layout (`~/amazon_ml/{dataset,code,utils}`)

In [ ]:
%%bash
cd ~/amazon_ml
[ -d dataset/train ] || unzip -q -o ~/dataset.zip -d .
ls -la dataset/train dataset/test code/business_entity_resolution/src utils

## 4. Run configuration
`N_SHARDS` = number of extra 300k-S1 training shards (1 → v9 size, 2 → v10, 3 → v11). Chosen automatically from RAM.
`N_FOLDS` = out-of-fold models for stage 2 (3 is what the best local model used).

In [ ]:
import os, psutil
ram_gb = psutil.virtual_memory().total / 1e9
N_SHARDS = 3 if ram_gb >= 100 else (2 if ram_gb >= 48 else 1)
N_FOLDS = 3
os.environ["N_SHARDS"], os.environ["N_FOLDS"] = str(N_SHARDS), str(N_FOLDS)
print(f"RAM {ram_gb:.0f} GB, {os.cpu_count()} vCPU  ->  N_SHARDS={N_SHARDS} ({300*N_SHARDS}k extra S1), N_FOLDS={N_FOLDS}")

## 5. Write the driver script and the leaderboard-rule script

In [ ]:
from pathlib import Path
root = Path.home() / "amazon_ml"

(root / "run_all.sh").write_text(r"""#!/bin/bash
# Resumable end-to-end driver: each finished stage leaves logs/<stage>.done
set -euo pipefail
ROOT=$HOME/amazon_ml; PKG=$ROOT/code/business_entity_resolution; LOG=$ROOT/logs
mkdir -p $LOG; source $HOME/venv/bin/activate
export PYTHONUNBUFFERED=1 PYTHONIOENCODING=utf-8 PYTHONUTF8=1
N_SHARDS=${N_SHARDS:-2}; N_FOLDS=${N_FOLDS:-3}; V=$((8 + N_SHARDS))
cd $PKG
stage () { [ -f $LOG/$1.done ] && { echo "skip $1 (done)"; return 0; }; echo "[$(date +%T)] start $1"; shift; "$@"; }
stage 1_train     bash -c "python -W ignore -m src.pipeline --mode train --n-train 150000 --n-val 30000 > $LOG/1_train.log 2>&1 && touch $LOG/1_train.done"
stage 2_bigtrain  bash -c "python -W ignore -m src.bigtrain $N_SHARDS $N_FOLDS > $LOG/2_bigtrain.log 2>&1 && touch $LOG/2_bigtrain.done"
stage 3_predict   bash -c "cp $ROOT/work/artifacts_v$V/lgb_matcher.txt $ROOT/work/artifacts_v$V/lgb_stage2.txt $ROOT/work/artifacts_v$V/policy.json artifacts/ && python -W ignore -m src.pipeline --mode predict --output-dir $ROOT/output > $LOG/3_predict.log 2>&1 && touch $LOG/3_predict.done"
cd $ROOT
stage 4_rule      bash -c "python -W ignore apply_rule.py > $LOG/4_rule.log 2>&1 && touch $LOG/4_rule.done"
python utils/validate_submission.py --matching output/matching_results.tsv --candidate output/candidate_pairs.tsv --test-dir dataset/test --check-ids > $LOG/5_validate.log 2>&1 || true
python utils/validate_submission.py --matching output/matching_results_LBRULE.tsv --candidate output/candidate_pairs.tsv --test-dir dataset/test --check-ids >> $LOG/5_validate.log 2>&1 || true
echo "[$(date +%T)] ALL DONE" | tee $LOG/ALL.done
""")

(root / "apply_rule.py").write_text(r"""
# Leaderboard-tuned decision rule on the stage-2 test scores:
#   each S2/S3 record -> its best S1 only (exclusivity)
#   keep a pair if p >= EXTRA, or if it is its S1's single best candidate with p >= TOP
import numpy as np, polars as pl
W, OUT, EXTRA, TOP = "work", "output", 0.82, 0.55
sc = pl.read_parquet(f"{W}/test_final_scores.parquet")
s1 = pl.read_parquet(f"{W}/test_s1_norm.parquet", columns=["entity_id"])["entity_id"]
t = pl.concat([pl.read_parquet(f"{W}/test_{p}_norm.parquet", columns=["entity_id"]) for p in ("s2", "s3")])["entity_id"]
sc = sc.filter(pl.col("p") == pl.col("p").max().over("t_idx"))
sc = sc.with_columns(pl.col("p").rank("ordinal", descending=True).over("s1_idx").alias("rk"))
sel = sc.filter((pl.col("p") >= EXTRA) | ((pl.col("rk") == 1) & (pl.col("p") >= TOP))).select("s1_idx", "t_idx").unique(maintain_order=True)
g = (sel.with_columns(pl.Series("t", t.gather(sel["t_idx"])))
     .group_by("s1_idx").agg(pl.col("t").unique().sort().str.join(",").alias("matched_entity_ids")))
base = pl.DataFrame({"s1_idx": np.arange(len(s1), dtype=np.int32), "source1_entity_id": s1})
out = (base.join(g.with_columns(pl.col("s1_idx").cast(pl.Int32)), on="s1_idx", how="left")
       .select("source1_entity_id", pl.col("matched_entity_ids").fill_null("")))
out.write_csv(f"{OUT}/matching_results_LBRULE.tsv", separator="\t", quote_style="never")
print("links:", sel.height, "| S1 with >=1 match:", sel["s1_idx"].n_unique(), "/", len(s1))
""")
print("written:", root / "run_all.sh", "and", root / "apply_rule.py")

## 6. Launch (safe to re-run: resumes from the last finished stage)

In [ ]:
%%bash
cd ~/amazon_ml && mkdir -p logs
nohup bash run_all.sh > logs/run_all.log 2>&1 &
sleep 3; cat logs/run_all.log

## 7. Status — re-run this cell any time
Expected on m7i.8xlarge: stage 1 ≈ 60 min · stage 2 ≈ 90–120 min · stage 3 ≈ 60–75 min · stage 4 + validation ≈ 10 min.

In [ ]:
%%bash
cd ~/amazon_ml
echo "=== driver"; cat logs/run_all.log
for f in logs/1_train.log logs/2_bigtrain.log logs/3_predict.log logs/4_rule.log; do
  [ -f $f ] && { echo "=== $f"; grep -vE "features\] [0-9]|valid_0" $f | tail -4; }
done
echo "=== held-out scores"; grep -h "\[val\]" logs/*.log 2>/dev/null || true
echo "=== machine"; uptime; free -g | head -2

## 8. Results
* `output/matching_results_LBRULE.tsv` — **upload this one first** (leaderboard-tuned rule).
* `output/matching_results.tsv` — same scores with the held-out-tuned threshold (for comparison).
* `output/candidate_pairs.tsv` — the candidate set (required in the final zip).

Held-out bar to beat on the same 30k S1: v9 **0.98527**, v10 stage 1 + v9 stage 2 **0.98550** (LB rule).

In [ ]:
%%bash
cd ~/amazon_ml
cat logs/5_validate.log 2>/dev/null | grep -E "PASS|FAIL|rows"
cat logs/4_rule.log 2>/dev/null
ls -la output/ work/artifacts_v* 2>/dev/null

## 9. Download the results to your PC, then STOP the instance
From Windows PowerShell:
```powershell
cd "C:\Users\surya\Downloads\amazon ml"
mkdir aws_output -Force
scp -i KEY.pem "ubuntu@IP:~/amazon_ml/output/*.tsv" aws_output\
scp -i KEY.pem -r "ubuntu@IP:~/amazon_ml/work/artifacts_v1*" aws_output\
```
Then in the AWS console: **EC2 → Instances → select → Instance state → Stop** (or Terminate once the files are safe).